In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [2]:
# experimental data
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_15779/1982994368.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_15779/1982994368.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [5]:
import numpy as np
from scipy.integrate import quad

limit = 10000000
abs_tol = 1e-15
rel_tol = 1e-15

# VEGAS-based full_int (replace previous full_int)
import numpy as np
try:
    import vegas
except Exception as e:
    raise ImportError("The 'vegas' package is required. Install with: pip install vegas") from e


In [6]:
import numpy as np
from scipy.integrate import quad
from numpy.polynomial.legendre import leggauss


# ============================================================
# Stable 2D integral:
#    - φ : quad (adaptive)
#    - x : Gauss–Legendre (deterministic, stable)
# ============================================================

def full_integral_quad(q2, mg, a1, a2, m2_func, sqrt_s, Nx=80):

    # Gauss-Legendre nodes for x ∈ [0,1]
    x_nodes, x_weights = leggauss(Nx)
    x_nodes = 0.5 * (x_nodes + 1.0)        # map from [-1,1] to [0,1]
    x_weights *= 0.5

    total_real = 0.0
    total_imag = 0.0
    total_err2 = 0.0

    for x, w in zip(x_nodes, x_weights):
        k = sqrt_s * x

        # integrand in φ
        def f(phi):
            val = T_1(k, q2, phi, mg, a1, a2, m2_func) \
                - T_2(k, q2, phi, mg, a1, a2, m2_func)
            return k * sqrt_s * val

        # real part
        r, er = quad(lambda ph: np.real(f(ph)), 0, 2*np.pi,
                     epsabs=1e-4, epsrel=1e-4)
        total_real += w * r
        total_err2 += (w * er)**2

        # imaginary part
        im, ei = quad(lambda ph: np.imag(f(ph)), 0, 2*np.pi,
                      epsabs=1e-4, epsrel=1e-4)
        total_imag += w * im
        total_err2 += (w * ei)**2

    T_value = total_real + 1j * total_imag
    T_error = np.sqrt(total_err2)

    return T_value, T_error


# ============================================================
# Differential cross section wrapper
# ============================================================

def get_dif_sigma_quad(epsilon, mg, a1, a2, mg_model, q2,
                       sqrt_s=7000.0, scale=1.0, Nx=2000):

    T_value, T_error = full_integral_quad(q2, mg, a1, a2, mg_model, sqrt_s, Nx)

    s = sqrt_s**2
    amp = amp_calculation(T_value, s, epsilon, -q2)
    dsigma = differential_sigma(amp, s) * scale

    if abs(T_value) > 0:
        dsigma_error = abs(dsigma) * 2 * (T_error / abs(T_value))
    else:
        dsigma_error = np.inf

    return {
        "q2": np.float64(q2),
        "T_value": T_value,
        "T_error": np.float64(T_error),
        "dsigma": dsigma,
        "dsigma_error": np.float64(dsigma_error)
    }


In [7]:
# # =============================================================
# #  PLOT BORN SIGMA TOT BORN
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 6000
max_sqrt_s = 13010
step = 800

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))


/tmp/ipykernel_15779/3011614697.py:6: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data_sigma_tot_atlas = pd.read_csv(


In [8]:
def get_sigma_tot_quad(epsilon, mg, a1, a2, mg_model,
                       start_sqrt_s=10,
                       max_sqrt_s=13000,
                       step=100,
                       Nx=80):
    """
    Compute sigma_tot(s) using the same quad-based 2D integral
    used for differential cross-sections.
    """

    lst_sqrt_s = []
    lst_sigma_tot = []
    lst_sigma_tot_error = []
    lst_relative_error = []
    lst_integral_values = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s + step * 0.5:

        s = sqrt_s**2
        t = 0.0   # σ_tot uses forward amplitude

        # ------------------------------------------------------------
        # 1) Perform 2D integral at t = 0 using the quad integrator
        # ------------------------------------------------------------
        T_value, T_error = full_integral_quad(
            q2=0.0,
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=mg_model,
            sqrt_s=sqrt_s,
            Nx=Nx
        )

        lst_integral_values.append(T_value)

        # ------------------------------------------------------------
        # 2) Optical theorem: σ_tot = Im A(s, t=0) / s
        # ------------------------------------------------------------
        born_amp = amp_calculation(T_value, s, epsilon, t)
        sigma_value = sigma_tot(born_amp, s)

        # ------------------------------------------------------------
        # 3) Error propagation
        #     Amplitude → cross section (linear)
        # ------------------------------------------------------------
        if abs(T_value) > 0:
            rel_T_err = T_error / abs(T_value)
            sigma_err = abs(sigma_value) * rel_T_err
        else:
            rel_T_err = np.inf
            sigma_err = np.inf

        lst_sqrt_s.append(sqrt_s)
        lst_sigma_tot.append(sigma_value)
        lst_sigma_tot_error.append(sigma_err)
        lst_relative_error.append(rel_T_err)

        # ------------------------------------------------------------
        # 4) Minimal console output
        # ------------------------------------------------------------
        print("─" * 80)
        print(f"√s = {sqrt_s:.6f}")
        print(f"Integral value (T):      {T_value:+.10e}")
        print(f"Integral error (quad):    {T_error:.3e}")
        print(f"σ_tot:                    {sigma_value:.6e} ± {sigma_err:.3e}")
        print("─" * 80)

        sqrt_s += step

    # ------------------------------------------------------------
    # Return results
    # ------------------------------------------------------------
    return {
        "sqrt_s": np.array(lst_sqrt_s),
        "sigma_tot": np.array(lst_sigma_tot),
        "sigma_tot_error": np.array(lst_sigma_tot_error),
        "relative_error": np.array(lst_relative_error),
        "integral_values": np.array(lst_integral_values),
    }


In [9]:
import numpy as np
from scipy.integrate import quad
from scipy.special import j0


# ============================================================
# χ(b) using quad + your quad-based amplitude integral
# ============================================================

def chi_of_b(b, epsilon, mg, a1, a2, mg_model,
             sqrt_s=7000,
             q_max=20.0,
             Nx_amp=80):
    """
    Computes χ(b) = (1/s) ∫_0^{q_max} dq  q J0(bq) A_Born(s, t = -q^2)
    using:
        - quad for oscillatory q-integral
        - full_integral_quad for the amplitude
    """

    s = sqrt_s**2

    # integrand in q
    def integrand(q):

        q2 = q*q
        t = -q2

        # amplitude T(q) from 2D integral
        T_value, T_error = full_integral_quad(
            q2=q2,
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=mg_model,
            sqrt_s=sqrt_s,
            Nx=Nx_amp
        )

        # Born amplitude
        A_born = amp_calculation(T_value, s, epsilon, t)

        # eikonal integrand
        return (q * j0(b*q) * A_born) / s

    # real part
    real_part, real_err = quad(lambda q: np.real(integrand(q)),
                               0, q_max, epsabs=1e-5, epsrel=1e-5)

    # imaginary part
    imag_part, imag_err = quad(lambda q: np.imag(integrand(q)),
                               0, q_max, epsabs=1e-5, epsrel=1e-5)

    chi_value = real_part + 1j*imag_part
    chi_error = np.sqrt(real_err**2 + imag_err**2)

    return chi_value, chi_error


In [12]:
sqrt_s = 7000
eps  = ensemble_parameters["atlas"]["pl"]["epsilon"]
mg   = ensemble_parameters["atlas"]["pl"]["mg"]
a1   = ensemble_parameters["atlas"]["pl"]["a1"]
a2   = ensemble_parameters["atlas"]["pl"]["a2"]
mdl  = m2_pl

lst_b_integration = np.linspace(0, 15, 30)

lst_chi = []

for b_val in lst_b_integration:
    chi_val, chi_err = chi_of_b(
        b=b_val,
        epsilon=eps,
        mg=mg,
        a1=a1,
        a2=a2,
        mg_model=mdl,
        sqrt_s=7000,
        q_max=20.0,
        Nx_amp=90
    )

    lst_chi.append(chi_val)

    print(f"b = {b_val:.2f},  χ(b) = {chi_val},  Im[χ(b)] = {np.imag(chi_val):.6f}, chi_err = {chi_err:.6f}")


b = 0.00,  χ(b) = 3.9341190365924845j,  Im[χ(b)] = 3.934119, chi_err = 0.000004
b = 0.52,  χ(b) = 3.905396942475148j,  Im[χ(b)] = 3.905397, chi_err = 0.000004
b = 1.03,  χ(b) = 3.820396826544151j,  Im[χ(b)] = 3.820397, chi_err = 0.000006


KeyboardInterrupt: 